In [1]:
!pip install librosa --quiet

In [2]:
import os, numpy as np, librosa, tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("carlthome/gtzan-genre-collection")

print("Path to dataset files:", path)

100%|██████████| 1.14G/1.14G [00:31<00:00, 38.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/carlthome/gtzan-genre-collection/versions/1


In [4]:
import os

data_dir = '/root/.cache/kagglehub/datasets/carlthome/gtzan-genre-collection/versions/1'
os.listdir(data_dir)


['genres']

In [5]:
data_dir = '/root/.cache/kagglehub/datasets/carlthome/gtzan-genre-collection/versions/1/genres'


In [6]:
def extract_features(file_path, max_pad_len=130):
    try:
        audio, sr = librosa.load(file_path, res_type='kaiser_fast', duration=30)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        pad_width = max_pad_len - mfccs.shape[1]
        if pad_width > 0:
            mfccs = np.pad(mfccs, pad_width=((0,0),(0,pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]
        return mfccs
    except Exception as e:
        print("Error:", file_path, e)
        return None

X, y, labels = [], [], os.listdir(data_dir)
print("Extracting MFCCs...")
for i, label in enumerate(labels):
    folder = os.path.join(data_dir, label)
    for f in tqdm(os.listdir(folder), desc=label):
        fp = os.path.join(folder, f)
        features = extract_features(fp)
        if features is not None:
            X.append(features)
            y.append(i)

X, y = np.array(X), np.array(y)
print("✅ Features shape:", X.shape)

Extracting MFCCs...


country: 100%|██████████| 100/100 [00:05<00:00, 16.75it/s]

✅ Features shape: (1000, 40, 130)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Reshape the input data to be 3D for the LSTM layer - This reshaping is redundant
# X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
# X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

In [8]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(labels), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 40, 128)        │       132,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,826 (729.79 KB)

 Trainable params: 186,826 (729.79 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                    epochs=20, batch_size=32)


Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.1081 - loss: 2.3182 - val_accuracy: 0.1150 - val_loss: 2.2741
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2200 - loss: 2.2022 - val_accuracy: 0.1700 - val_loss: 2.2261
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3251 - loss: 2.0577 - val_accuracy: 0.2050 - val_loss: 2.1727
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3949 - loss: 1.8425 - val_accuracy: 0.2550 - val_loss: 2.1320
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5327 - loss: 1.5240 - val_accuracy: 0.2550 - val_loss: 2.2173
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5794 - loss: 1.3003 - val_accuracy: 0.2650 - val_loss: 2.3025
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6914 - loss: 1.0368 - val_accuracy: 0.2400 - val_loss: 2.4594
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7480 - loss: 0.8346 - val_accuracy: 0.2450 - v

In [10]:
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2434 - loss: 4.0047 

✅ Test Accuracy: 25.00%
